# CS2309 — Precision disk/VRAM eval (fp16 disk + fp4)

**Phase A (Mac):** convert fp16 + eval `baseline_fp32` vs `fp16_disk`.
**Phase B (Colab T4):** eval thêm `fp4_from_fp16`.

### Nguồn weights fp16 — chọn một

| `USE_DRIVE` | Hành vi |
|---|---|
| `True` | Mount Google Drive → gắn `swiftedit_weights_fp16` đã upload |
| `False` | **Không** mount Drive → `setup_colab` tải weights **fp32** (~9.6GB) → `convert_weights_fp16.py` sinh tree fp16 trên `/content` |

### Repo private (giống notebook test / phase3 / webui)

- `USE_PRIVATE_REPO = True`: Colab → 🔑 **Secrets** → `GITHUB_TOKEN` = PAT, bật Notebook access.
- Extension: Secrets thường timeout → cell hỏi **getpass** (nhập PAT, không ghi vào file).
- Chỉnh `REPO_SLUG` / `USE_DRIVE` / `USE_PRIVATE_REPO` ở cell cấu hình.

### Output tải về so báo cáo

`experimental_data/precision_disk_vram_<date>/bundle.zip`

### ⓪ Cấu hình (USE_DRIVE / USE_PRIVATE_REPO / path Drive / repo)

In [ ]:
# --- Tùy chọn chính ---
USE_DRIVE = True  # False = không mount Drive; tải fp32 + convert fp16 trên runtime
USE_PRIVATE_REPO = True  # True: cần GITHUB_TOKEN (Secrets hoặc getpass)

REPO_SLUG = "NguyenKz/CS2309.CH201"
COLAB_REPO_DIR = "/content/CS2309.CH201"
# Chỉ dùng khi USE_DRIVE=True (đổi nếu upload chỗ khác)
DRIVE_FP16 = "/content/drive/MyDrive/CS2309/swiftedit_weights_fp16"

N_IMAGES = 4
EDITS_PER_IMAGE = 3

print("USE_DRIVE:", USE_DRIVE)
print("USE_PRIVATE_REPO:", USE_PRIVATE_REPO)
print("REPO_SLUG:", REPO_SLUG)
print("DRIVE_FP16:", DRIVE_FP16 if USE_DRIVE else "(không dùng)")

### ① Clone repo + GPU (+ mount Drive nếu bật) + GITHUB_TOKEN

In [ ]:
import getpass
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

COLAB_REPO_DIR = Path(COLAB_REPO_DIR)
DRIVE_FP16 = Path(DRIVE_FP16)

_COLAB_GPU_ERR = (
    "Colab chưa có GPU.\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Colab extension: Select Kernel → Colab → New Colab Server "
    "→ Hardware accelerator: GPU → T4, rồi Restart kernel"
)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK (nvidia-smi):", ", ".join(names))


def _colab_repo_url():
    """Giống notebook khác: Secrets trước; fallback getpass (extension)."""
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    token = None
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print(
            "Không lấy GITHUB_TOKEN từ Colab Secrets "
            f"(thường gặp trên extension).\nChi tiết: {e}"
        )
    if not token:
        print(
            "Nhập PAT (repo read). Không lưu vào notebook.\n"
            "Hoặc Colab web → 🔑 Secrets → GITHUB_TOKEN + Allow notebook access."
        )
        token = getpass.getpass("GITHUB_TOKEN (PAT): ").strip()
    if not token:
        raise RuntimeError(
            "Thiếu GITHUB_TOKEN. Thêm Secret hoặc nhập PAT, rồi chạy lại cell.\n"
            f"Hoặc đặt USE_PRIVATE_REPO=False nếu repo đã public: {public_url}"
        )
    return f"https://{token}@github.com/{REPO_SLUG}.git"


if IN_COLAB:
    _check_colab_gpu()

    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        print("Drive mounted.")
    else:
        print("Bỏ qua mount Drive — sẽ tải fp32 + convert fp16 trên runtime.")

    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        REPO_URL = _colab_repo_url()
        print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.chdir(PROJECT_ROOT)
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    print("Local — Drive / GITHUB_TOKEN không áp dụng; dùng weights trong repo.")

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("USE_DRIVE:", USE_DRIVE, "| USE_PRIVATE_REPO:", USE_PRIVATE_REPO)
if IN_COLAB and not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
    raise FileNotFoundError("Clone xong nhưng thiếu SwiftEdit/ — kiểm tra token / REPO_SLUG.")

### ② Setup pip + tải weights **fp32** + HF (VAE/CLIP)

`setup_colab.sh` tải `swiftedit_weights` (~9.6GB) nếu chưa có — cần cho `baseline_fp32`.
Khi `USE_DRIVE=False`, bước này là nguồn chính; sau đó cell ③ convert sang fp16.

In [ ]:
env = os.environ.copy()
env["COLAB_REPO_DIR"] = str(PROJECT_ROOT) if IN_COLAB else env.get("COLAB_REPO_DIR", "")
setup_sh = PROJECT_ROOT / "scripts" / ("setup_colab.sh" if IN_COLAB else "setup_macos.sh")
print("Chạy:", setup_sh)
subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "torchmetrics"],
    check=True,
)
WP32 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"
print("fp32 weights OK:", (WP32 / "sbv2_0.5").is_dir())

### ③ Lấy `swiftedit_weights_fp16`

- **`USE_DRIVE=True`:** symlink từ Drive (nhanh nếu đã upload).
- **`USE_DRIVE=False`:** chạy `convert_weights_fp16.py` từ tree fp32 vừa tải (~vài phút + ~5GB thêm trên disk runtime).

In [ ]:
WP16 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights_fp16"

if IN_COLAB and USE_DRIVE:
    link_py = PROJECT_ROOT / "scripts" / "link_weights_fp16_drive.py"
    cmd = [sys.executable, str(link_py), "--drive-dir", str(DRIVE_FP16)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
elif not (WP16 / "sbv2_0.5").is_dir():
    convert_py = PROJECT_ROOT / "scripts" / "convert_weights_fp16.py"
    cmd = [
        sys.executable,
        str(convert_py),
        "--src",
        str(WP32),
        "--dst",
        str(WP16),
    ]
    print("Convert fp32 → fp16 (không dùng Drive):")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print("Đã có", WP16, "— bỏ qua convert/link.")

if not (WP16 / "sbv2_0.5").is_dir():
    raise FileNotFoundError(
        f"Thiếu fp16 weights tại {WP16}. "
        "USE_DRIVE=True: kiểm tra DRIVE_FP16 đã upload/giải nén. "
        "USE_DRIVE=False: kiểm tra convert và dung lượng đĩa."
    )
print("fp16 OK:", WP16)
print("Nguồn:", "Google Drive" if (IN_COLAB and USE_DRIVE) else "convert từ fp32 / local")

### ④ Eval + tải `bundle.zip`

Colab: `baseline_fp32,fp16_disk,fp4_from_fp16`. Local Mac: bỏ fp4.
Prompt lấy từ `mapping_file.json` (PIE-Bench-smoke) — không gán prompt xe đạp cho ảnh người.

In [ ]:
configs = (
    "baseline_fp32,fp16_disk,fp4_from_fp16" if IN_COLAB else "baseline_fp32,fp16_disk"
)
eval_py = PROJECT_ROOT / "scripts" / "run_precision_disk_vram_eval.py"
cmd = [
    sys.executable,
    str(eval_py),
    "--configs",
    configs,
    "--n-images",
    str(N_IMAGES),
    "--edits-per-image",
    str(EDITS_PER_IMAGE),
    "--weights-fp32",
    str(WP32),
    "--weights-fp16",
    str(WP16),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

bundles = sorted((PROJECT_ROOT / "experimental_data").glob("precision_disk_vram_*/bundle.zip"))
print("Bundles:", bundles)
if IN_COLAB and bundles:
    from google.colab import files

    print("Download:", bundles[-1])
    files.download(str(bundles[-1]))